In [0]:
%python
# --- Parameters ---

# --- Read Stream from Kafka ---
from pyspark.dbutils import dbutils
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType
)

confluentBootstrapServers = dbutils.get("confluent_cloud_busradar_cluster", "confluentBootstrapServers")
confluentTopicName = dbutils.get("confluent_cloud_busradar_cluster", "confluentTopicName")
confluentApiKey = dbutils.get("confluent_cloud_busradar_cluster", "confluentApiKey")
confluentSecret = dbutils.get("confluent_cloud_busradar_cluster", "confluentSecret")



bus_schema = StructType([
    StructField("fahrtbezeichner", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("linientext", StringType(), True),
    StructField("linienid", StringType(), True),
    StructField("richtungstext", StringType(), True),
    StructField("richtungsid", StringType(), True),
    StructField("delay", StringType(), True),
    StructField("sequenz", StringType(), True),
    StructField("fahrtstatus", StringType(), True),
    StructField("fahrzeugid", StringType(), True),
    StructField("nachhst", StringType(), True),
    StructField("akthst", StringType(), True),
    StructField("starthst", StringType(), True),
    StructField("zielhst", StringType(), True),
    StructField("betriebstag", StringType(), True),
    StructField("visfahrplanlagezst", LongType(), True),
    StructField("ankunftziel", LongType(), True),
    StructField("abfahrtstart", LongType(), True),
])

kafka_options = {
    "kafka.bootstrap.servers": confluentBootstrapServers,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{confluentApiKey}" password="{confluentSecret}";',
    "subscribe": confluentTopicName,
    "startingOffsets": "latest",
}

df_raw = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load())

# --- Parse ---
df_parsed = (
    df_raw
    .select(
        from_json(col("value").cast("string"), bus_schema).alias("data"),
        col("timestamp").alias("kafka_timestamp"),
    )
    .select("data.*", "kafka_timestamp")
    .withColumn("_loaded_at", current_timestamp())
)

# --- Write to Delta ---
(
    df_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/bus_radar_dev/bronze/checkpoints/raw_positions")
    .trigger(availableNow=True)
    .toTable("bus_radar_dev.bronze.raw_positions")
    .awaitTermination()
)


